# Week 11 — Search: Full-Text, Vector, and Hybrid

**IT 2012 — Unstructured Data**

This week we make our cleaned documents **searchable**. We'll implement three search approaches:

1. **Keyword search** — match exact words (fast, but misses synonyms)
2. **Semantic search** — match meaning using embeddings (understands synonyms, but slower)
3. **Hybrid search** — combine both for the best results

By the end of this notebook, you'll be able to type a natural language question and find relevant documents — even when the query words don't appear in the text.

## 1. Setup and Load Cleaned Data

First, install the new libraries for this week:

```bash
pip install sentence-transformers chromadb
```

`sentence-transformers` turns text into numerical vectors (embeddings).  
`chromadb` is a lightweight vector database that stores and searches embeddings.

In [3]:
import sys
!{sys.executable} -m pip install sentence-transformers chromadb



  Using cached overrides-7.7.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached websocket_client-1.9.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl.metadata (11 kB)
  Using cached importlib_metadata-8.7.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached oauthlib-3.3.1-py3-none-any.whl.metadata (7.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 522.4 kB/s  0:00:00m-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 1.7 MB/s  0:00:13m0:00:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 4.5 MB/s  0:00:02m0:00:0100:01
Using cached

In [4]:
# Standard libraries
import json
import re
from collections import Counter

# Data science (from Weeks 8-9)
import numpy as np
import pandas as pd

# NEW this week
from sentence_transformers import SentenceTransformer
import chromadb

print("All imports successful!")

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports successful!


In [5]:
# Load our cleaned documents from Week 9
with open("data/cleaned_documents.json", "r", encoding="utf-8") as f:
    documents = json.load(f)

df = pd.DataFrame(documents)
print(f"Loaded {len(df)} cleaned documents")
print(f"\nSource types:\n{df['source_type'].value_counts().to_string()}")
print(f"\nLanguages:\n{df['language'].value_counts().to_string()}")
print(f"\nSample document:")
print(f"  Title: {df.iloc[0]['title']}")
print(f"  Text:  {df.iloc[0]['extracted_text'][:100]}...")

Loaded 43 cleaned documents

Source types:
source_type
pdf      18
web      10
audio     8
image     7

Languages:
language
en    31
bs    12

Sample document:
  Title: Sarajevo Tourism Report 2024
  Text:  Sarajevo welcomed over 1.2 million tourists in 2024, a 15% increase from the previous year. The Old ...


## 2. Why Search Matters for Unstructured Data

With 43 documents, you could read them all manually. But real pipelines have **thousands or millions** of documents. You need a way to find the right document quickly.

### The Problem with Simple Approaches

| Approach | Works for | Fails for |
|----------|-----------|-----------|
| `Ctrl+F` (exact match) | Finding "ChromaDB" in text | Finding all documents about databases |
| File names | Well-named files | "meeting_notes_v3_final.pdf" |
| Folder structure | Organized collections | Cross-cutting queries |

### Three Levels of Search

1. **Keyword / Full-text search** — Find documents containing specific words  
   - ✅ Fast, predictable, good for exact terms (invoice numbers, names)
   - ❌ Misses synonyms, doesn't understand context

2. **Semantic / vector search** — Find documents with similar *meaning*  
   - ✅ Understands synonyms, handles natural language queries  
   - ❌ Slower, requires embedding model, can be unpredictable

3. **Hybrid search** — Combine both approaches  
   - ✅ Best of both worlds  
   - ❌ More complex to implement

## 3. Keyword Search — The Baseline

Let's start with the simplest approach: searching for exact word matches.  
This is what `Ctrl+F` does, but across all documents.

In [6]:
def keyword_search(documents_df, query, top_k=5):
    """
    Simple keyword search: count how many query words appear in each document.
    Returns top-k results ranked by match count.
    """
    # Tokenize query into individual words, lowercase
    query_words = set(query.lower().split())
    
    results = []
    for idx, row in documents_df.iterrows():
        text = row["extracted_text"].lower()
        # Count how many query words appear in the document
        matches = sum(1 for word in query_words if word in text)
        if matches > 0:
            results.append({
                "title": row["title"],
                "source_type": row["source_type"],
                "matches": matches,
                "total_query_words": len(query_words),
                "score": matches / len(query_words),  # Fraction of query words found
                "text_preview": row["extracted_text"][:120] + "..."
            })
    
    # Sort by score (fraction of query words matched)
    results.sort(key=lambda x: x["score"], reverse=True)
    return results[:top_k]

In [7]:
# Test keyword search
query = "database security vulnerabilities"
results = keyword_search(df, query)

print(f"Keyword search: '{query}'")
print(f"Found {len(results)} matching documents\n")

for i, r in enumerate(results, 1):
    print(f"{i}. [{r['source_type']}] {r['title']}")
    print(f"   Score: {r['score']:.0%} ({r['matches']}/{r['total_query_words']} words matched)")
    print(f"   Preview: {r['text_preview']}")
    print()

Keyword search: 'database security vulnerabilities'
Found 5 matching documents

1. [pdf] Security Audit Report October 2024
   Score: 100% (3/3 words matched)
   Preview: Critical findings: 1) SQL injection vulnerability in the student portal login form - CVSS score 9.1. 2) Outdated SSL cer...

2. [pdf] Meeting Minutes - IT Department
   Score: 67% (2/3 words matched)
   Preview: IT Department meeting held on October 15, 2024. Attendees: Dzelila, Marko, Amir, Lejla. Agenda: 1) Server migration time...

3. [pdf] Invoice INV-00123
   Score: 33% (1/3 words matched)
   Preview: Invoice number INV-00123 for services rendered in October 2024. Client: TechBridge d.o.o. Sarajevo. Services: Web applic...

4. [pdf] Annual Budget Proposal 2025
   Score: 33% (1/3 words matched)
   Preview: The proposed budget for fiscal year 2025 allocates 1.8 million KM to technology infrastructure, representing a 20% incre...

5. [pdf] Contract Agreement v3
   Score: 33% (1/3 words matched)
   Preview: Service a

### Where Keyword Search Fails

Let's try a query where the exact words DON'T appear in any document:

In [8]:
# The word "automobile" doesn't appear anywhere, but "car" does in some contexts
query = "cloud computing migration"
results = keyword_search(df, query)
print(f"Keyword search: '{query}'")
print(f"Found {len(results)} matching documents\n")
for i, r in enumerate(results, 1):
    print(f"{i}. [{r['source_type']}] {r['title']} — Score: {r['score']:.0%}")
    print(f"   {r['text_preview']}")
    print()

print("---")
# Now try with different words that MEAN the same thing
query2 = "moving servers to the cloud infrastructure"
results2 = keyword_search(df, query2)
print(f"\nKeyword search: '{query2}'")
print(f"Found {len(results2)} matching documents\n")
for i, r in enumerate(results2, 1):
    print(f"{i}. [{r['source_type']}] {r['title']} — Score: {r['score']:.0%}")

print("\n→ Different words, same intent — keyword search gives different results!")

Keyword search: 'cloud computing migration'
Found 5 matching documents

1. [pdf] Meeting Minutes - IT Department — Score: 67%
   IT Department meeting held on October 15, 2024. Attendees: Dzelila, Marko, Amir, Lejla. Agenda: 1) Server migration time...

2. [pdf] Annual Budget Proposal 2025 — Score: 67%
   The proposed budget for fiscal year 2025 allocates 1.8 million KM to technology infrastructure, representing a 20% incre...

3. [pdf] Server Migration Plan — Score: 67%
   Phase 1: Inventory all current servers and services (2 weeks). Phase 2: Set up cloud infrastructure on Azure (1 week). P...

4. [pdf] Contract Agreement v3 — Score: 33%
   Service agreement between International Burch University and CloudServe d.o.o. for cloud hosting services. Term: January...

5. [pdf] Python Workshop Handout — Score: 33%
   Introduction to Python for data processing. Python is the most popular language for data science and machine learning. K...

---

Keyword search: 'moving servers to the cloud 

## 4. Text Embeddings — Turning Words Into Numbers

### The Core Idea

An **embedding** is a list of numbers (a vector) that represents the *meaning* of text.

- `"cat"` → `[0.21, -0.45, 0.82, ..., 0.13]`  (384 numbers)
- `"kitten"` → `[0.23, -0.44, 0.80, ..., 0.15]`  (very similar numbers!)
- `"database"` → `[-0.67, 0.12, -0.34, ..., 0.89]`  (very different numbers)

**Similar meanings → similar vectors.**

This is what makes semantic search possible: instead of matching words, we compare the *meaning* of the query with the *meaning* of each document.

### Loading an Embedding Model

We'll use `all-MiniLM-L6-v2` — a compact model that produces 384-dimensional embeddings.  
For multilingual text (our dataset has both English and Bosnian), `paraphrase-multilingual-MiniLM-L12-v2` is a better choice.

In [9]:
# Load the embedding model
# First run downloads the model (~80 MB) — subsequent runs use cache
model = SentenceTransformer("all-MiniLM-L6-v2")

# Check model details
print(f"Model: all-MiniLM-L6-v2")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")
print(f"Max input length: {model.max_seq_length} tokens")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14886.23it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model: all-MiniLM-L6-v2
Embedding dimension: 384
Max input length: 256 tokens


/var/folders/pg/9y8wr59503l5z85h5p_mynlr0000gn/T/ipykernel_512/1681223326.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")


In [10]:
# Generate an embedding for a single sentence
text = "Sarajevo is the capital of Bosnia and Herzegovina"
embedding = model.encode(text)

print(f"Input text: '{text}'")
print(f"Embedding shape: {embedding.shape}")
print(f"First 10 values: {embedding[:10].round(4)}")
print(f"Min: {embedding.min():.4f}, Max: {embedding.max():.4f}")
print(f"Norm (length): {np.linalg.norm(embedding):.4f}")

Input text: 'Sarajevo is the capital of Bosnia and Herzegovina'
Embedding shape: (384,)
First 10 values: [ 0.0594  0.0804 -0.0394  0.0962 -0.0242  0.0038 -0.037  -0.0062  0.0397
 -0.0055]
Min: -0.1438, Max: 0.1424
Norm (length): 1.0000


### Embeddings Capture Meaning

Let's see how similar sentences get similar embeddings:

In [11]:
# Encode several sentences
sentences = [
    "The server needs to be migrated to the cloud",   # Group 1: cloud/servers
    "We are moving our infrastructure to Azure",       # Group 1
    "Cloud migration is scheduled for December",       # Group 1
    "Bosnian cuisine features grilled meat dishes",    # Group 2: food
    "Ćevapi are a traditional Bosnian food",           # Group 2
    "The restaurant serves local Balkan specialties",  # Group 2
    "Python is used for data science and machine learning",  # Group 3: tech
    "Data processing with pandas and NumPy",           # Group 3
]

embeddings = model.encode(sentences)
print(f"Encoded {len(sentences)} sentences → shape: {embeddings.shape}")
print(f"Each sentence is now a vector of {embeddings.shape[1]} numbers")

Encoded 8 sentences → shape: (8, 384)
Each sentence is now a vector of 384 numbers


## 5. Similarity Measures

How do we compare two embeddings? We need a **similarity measure**.

### Cosine Similarity

Measures the **angle** between two vectors (ignores length, focuses on direction).

- `1.0` = identical meaning
- `0.0` = completely unrelated  
- `-1.0` = opposite meaning (rare in practice)

This is the most commonly used measure for text embeddings.

### Formula

$$\text{cosine\_similarity}(A, B) = \frac{A \cdot B}{\|A\| \times \|B\|}$$

In [12]:
def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    return dot_product / (norm_a * norm_b)

# Compare sentence pairs
print("Similarity matrix (sentences from Section 4):\n")
print(f"{'':>45}", end="")
for i in range(len(sentences)):
    print(f"  S{i+1}", end="")
print()

for i, sent_i in enumerate(sentences):
    label = sent_i[:42] + "..." if len(sent_i) > 45 else sent_i
    print(f"S{i+1} {label:>42}", end="")
    for j in range(len(sentences)):
        sim = cosine_similarity(embeddings[i], embeddings[j])
        print(f" {sim:.2f}", end="")
    print()

Similarity matrix (sentences from Section 4):

                                               S1  S2  S3  S4  S5  S6  S7  S8
S1 The server needs to be migrated to the cloud 1.00 0.53 0.60 -0.07 -0.02 -0.02 0.08 0.05
S2  We are moving our infrastructure to Azure 0.53 1.00 0.49 -0.09 -0.03 -0.01 0.17 0.07
S3  Cloud migration is scheduled for December 0.60 0.49 1.00 -0.12 -0.02 -0.09 0.13 0.08
S4 Bosnian cuisine features grilled meat dishes -0.07 -0.09 -0.12 1.00 0.60 0.65 -0.02 0.03
S5      Ćevapi are a traditional Bosnian food -0.02 -0.03 -0.02 0.60 1.00 0.46 -0.03 -0.04
S6 The restaurant serves local Balkan special... -0.02 -0.01 -0.09 0.65 0.46 1.00 0.07 0.08
S7 Python is used for data science and machin... 0.08 0.17 0.13 -0.02 -0.03 0.07 1.00 0.41
S8      Data processing with pandas and NumPy 0.05 0.07 0.08 0.03 -0.04 0.08 0.41 1.00


In [13]:
# More readable: show similarity groups
print("=== Similarity within groups ===\n")

groups = {
    "Cloud/Servers (S1-S3)": [0, 1, 2],
    "Food/Cuisine (S4-S6)": [3, 4, 5],
    "Tech/Python (S7-S8)": [6, 7],
}

for group_name, indices in groups.items():
    sims = []
    for i in range(len(indices)):
        for j in range(i+1, len(indices)):
            sim = cosine_similarity(embeddings[indices[i]], embeddings[indices[j]])
            sims.append(sim)
    if sims:
        print(f"{group_name}: avg similarity = {np.mean(sims):.3f}")

print("\n=== Similarity between groups ===\n")

group_pairs = [
    ("Cloud/Servers", [0,1,2], "Food/Cuisine", [3,4,5]),
    ("Cloud/Servers", [0,1,2], "Tech/Python", [6,7]),
    ("Food/Cuisine", [3,4,5], "Tech/Python", [6,7]),
]

for name_a, idx_a, name_b, idx_b in group_pairs:
    sims = []
    for i in idx_a:
        for j in idx_b:
            sims.append(cosine_similarity(embeddings[i], embeddings[j]))
    print(f"{name_a} ↔ {name_b}: avg similarity = {np.mean(sims):.3f}")

print("\n→ Documents with similar topics cluster together in embedding space!")

=== Similarity within groups ===

Cloud/Servers (S1-S3): avg similarity = 0.540
Food/Cuisine (S4-S6): avg similarity = 0.568
Tech/Python (S7-S8): avg similarity = 0.411

=== Similarity between groups ===

Cloud/Servers ↔ Food/Cuisine: avg similarity = -0.051
Cloud/Servers ↔ Tech/Python: avg similarity = 0.097
Food/Cuisine ↔ Tech/Python: avg similarity = 0.016

→ Documents with similar topics cluster together in embedding space!


### Other Similarity Measures

| Measure | Formula | When to use |
|---------|---------|-------------|
| **Cosine similarity** | angle between vectors | Most common for text embeddings |
| **Dot product** | `a · b` | When embeddings are already normalized |
| **Euclidean distance** | `||a - b||` | When magnitude matters |

For **sentence-transformers** models, cosine similarity is the standard choice.

## 6. Building Semantic Search (from Scratch)

Before using ChromaDB, let's build semantic search manually to understand how it works.

The process:
1. Embed all documents (once, at index time)
2. Embed the query (at search time)  
3. Compute similarity between query and all documents
4. Return the top-k most similar

In [14]:
# Step 1: Embed all documents
texts = df["extracted_text"].tolist()
print(f"Embedding {len(texts)} documents...")

doc_embeddings = model.encode(texts, show_progress_bar=True)
print(f"Done! Shape: {doc_embeddings.shape}")

Embedding 43 documents...


Batches: 100%|██████████| 2/2 [00:00<00:00,  2.72it/s]

Done! Shape: (43, 384)


In [16]:
def semantic_search_manual(query, doc_embeddings, documents_df, top_k=5):
    """
    Manual semantic search:
    1. Embed the query
    2. Compute cosine similarity with all documents
    3. Return top-k results
    """
    # Embed the query
    query_embedding = model.encode(query)
    
    # Compute similarity with all documents
    similarities = []
    for i, doc_emb in enumerate(doc_embeddings):
        sim = cosine_similarity(query_embedding, doc_emb)
        similarities.append(sim)
    
    # Sort by similarity (highest first)
    ranked_indices = np.argsort(similarities)[::-1]
    
    results = []
    for idx in ranked_indices[:top_k]:
        results.append({
            "title": documents_df.iloc[idx]["title"],
            "source_type": documents_df.iloc[idx]["source_type"],
            "similarity": similarities[idx],
            "text_preview": documents_df.iloc[idx]["extracted_text"][:120] + "..."
        })
    
    return results

In [17]:
# Test semantic search with the same queries we used for keyword search
query = "database security vulnerabilities"
results = semantic_search_manual(query, doc_embeddings, df)

print(f"Semantic search: '{query}'")
print(f"Top {len(results)} results:\n")

for i, r in enumerate(results, 1):
    print(f"{i}. [{r['source_type']}] {r['title']}")
    print(f"   Similarity: {r['similarity']:.4f}")
    print(f"   Preview: {r['text_preview']}")
    print()

Semantic search: 'database security vulnerabilities'
Top 5 results:

1. [pdf] Security Audit Report October 2024
   Similarity: 0.5269
   Preview: Critical findings: 1) SQL injection vulnerability in the student portal login form - CVSS score 9.1. 2) Outdated SSL cer...

2. [pdf] Database Comparison Report
   Similarity: 0.3859
   Preview: Comparison of database solutions for the document management system. PostgreSQL: strong ACID compliance, excellent for s...

3. [web] avaz.ba - Sajam zapošljavanja Sarajevo
   Similarity: 0.1738
   Preview: U Sarajevu je održan sajam zapošljavanja na kojem je učestvovalo 40 kompanija iz IT sektora. Najtraženija zanimanja su b...

4. [image] handwritten_notes_lecture
   Similarity: 0.1590
   Preview: Data pipeline stages: 1. Acquire raw data from sources. 2. Store in appropriate format. 3. Extract and parse content. 4....

5. [image] whiteboard_photo
   Similarity: 0.1533
   Preview: Sprint planning - Week 42. Tasks: implement user authentication modu

In [18]:
# Now try the query that keyword search struggled with
query = "cloud computing migration"
results_semantic = semantic_search_manual(query, doc_embeddings, df)
results_keyword = keyword_search(df, query)

print(f"Query: '{query}'\n")
print("=" * 60)
print("KEYWORD SEARCH RESULTS:")
print("=" * 60)
for i, r in enumerate(results_keyword[:3], 1):
    print(f"  {i}. {r['title']} (score: {r['score']:.0%})")

print()
print("=" * 60)
print("SEMANTIC SEARCH RESULTS:")
print("=" * 60)
for i, r in enumerate(results_semantic[:3], 1):
    print(f"  {i}. {r['title']} (similarity: {r['similarity']:.4f})")

print("\n→ Semantic search understands the MEANING, not just the words!")

Query: 'cloud computing migration'

KEYWORD SEARCH RESULTS:
  1. Meeting Minutes - IT Department (score: 67%)
  2. Annual Budget Proposal 2025 (score: 67%)
  3. Server Migration Plan (score: 67%)

SEMANTIC SEARCH RESULTS:
  1. Server Migration Plan (similarity: 0.5886)
  2. Meeting Minutes - IT Department (similarity: 0.4397)
  3. Contract Agreement v3 (similarity: 0.4171)

→ Semantic search understands the MEANING, not just the words!


### 💡 Key Insight

Notice how semantic search finds the **Server Migration Plan** document even if the query uses "cloud computing" instead of the exact words in the document. The embedding model understands these concepts are related.

## 7. Vector Search with ChromaDB

Our manual approach works but doesn't scale. For larger collections, we need a **vector database**.

**ChromaDB** is a lightweight vector database that:
- Stores documents + embeddings + metadata
- Provides fast similarity search using optimized algorithms
- Supports metadata filtering  
- Can run embedded in your Python process (no separate server needed)

### Why not just use NumPy?

| | NumPy (manual) | ChromaDB |
|--|----------------|----------|
| Storage | RAM only | Persistent on disk |
| Speed at 1K docs | Fast | Fast |
| Speed at 1M docs | Slow (linear scan) | Fast (ANN index) |
| Metadata filtering | Manual | Built-in |
| Updates/deletes | Manual | Built-in |

In [19]:
# Create a ChromaDB client
# In-memory (for this demo):
client = chromadb.Client()

# For persistent storage (saves to disk):
# client = chromadb.PersistentClient(path="./chroma_db")

print(f"ChromaDB client created")
print(f"Type: {type(client).__name__}")

ChromaDB client created
Type: Client


In [20]:
# Create a collection (like a "table" in SQL)
# We specify our own embedding function using sentence-transformers
from chromadb.utils import embedding_functions

# Use the same model we've been using
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Create (or get if it already exists) the collection
collection = client.get_or_create_collection(
    name="visit_bosnia_docs",
    embedding_function=embedding_fn,
    metadata={"description": "Week 10 - VisitBosnia pipeline documents"}
)

print(f"Collection '{collection.name}' created")
print(f"Current document count: {collection.count()}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6885.55it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Collection 'visit_bosnia_docs' created
Current document count: 0


In [21]:
# Add all documents to the collection
# ChromaDB needs: ids, documents, and optionally metadatas

ids = [doc["_id"] for doc in documents]
texts = [doc["extracted_text"] for doc in documents]
metadatas = [
    {
        "source_type": doc["source_type"],
        "title": doc["title"],
        "language": doc["language"],
        "word_count": doc["word_count"],
        "confidence": doc["confidence"],
        "processing_date": doc["processing_date"],
    }
    for doc in documents
]

# Add in a single batch (ChromaDB handles embedding automatically)
collection.add(
    ids=ids,
    documents=texts,
    metadatas=metadatas,
)

print(f"Added {collection.count()} documents to the collection")

Added 43 documents to the collection


### Querying ChromaDB

The simplest query: pass text, get similar documents back.

In [22]:
# Basic semantic search with ChromaDB
results = collection.query(
    query_texts=["cloud computing migration"],
    n_results=5,
)

print("ChromaDB semantic search: 'cloud computing migration'\n")

for i in range(len(results["ids"][0])):
    doc_id = results["ids"][0][i]
    distance = results["distances"][0][i]
    metadata = results["metadatas"][0][i]
    text = results["documents"][0][i][:120]
    
    print(f"{i+1}. [{metadata['source_type']}] {metadata['title']}")
    print(f"   Distance: {distance:.4f} (lower = more similar)")
    print(f"   Preview: {text}...")
    print()

ChromaDB semantic search: 'cloud computing migration'

1. [pdf] Server Migration Plan
   Distance: 0.4114 (lower = more similar)
   Preview: Phase 1: Inventory all current servers and services (2 weeks). Phase 2: Set up cloud infrastructure on Azure (1 week). P...

2. [pdf] Meeting Minutes - IT Department
   Distance: 0.5603 (lower = more similar)
   Preview: IT Department meeting held on October 15, 2024. Attendees: Dzelila, Marko, Amir, Lejla. Agenda: 1) Server migration time...

3. [pdf] Contract Agreement v3
   Distance: 0.5829 (lower = more similar)
   Preview: Service agreement between International Burch University and CloudServe d.o.o. for cloud hosting services. Term: January...

4. [pdf] Annual Budget Proposal 2025
   Distance: 0.6083 (lower = more similar)
   Preview: The proposed budget for fiscal year 2025 allocates 1.8 million KM to technology infrastructure, representing a 20% incre...

5. [pdf] Docker Workshop Guide
   Distance: 0.7388 (lower = more similar)
   Preview:

### ⚠️ Distance vs Similarity

ChromaDB returns **distances** (lower = more similar), not similarities (higher = more similar).

- Default distance metric: **L2 (Euclidean distance)**
- Distance `0.0` = identical
- You can switch to cosine: `get_or_create_collection(..., metadata={"hnsw:space": "cosine"})`

## 8. Metadata Filtering

One of ChromaDB's most useful features: filter by metadata **before** vector search.

This narrows the search space — like saying "search only within PDF documents" or "only documents in English".

In [23]:
# Search only within PDF documents
results = collection.query(
    query_texts=["budget financial planning"],
    n_results=5,
    where={"source_type": "pdf"},  # ← metadata filter
)

print("Search: 'budget financial planning' (PDFs only)\n")
for i in range(len(results["ids"][0])):
    meta = results["metadatas"][0][i]
    dist = results["distances"][0][i]
    print(f"{i+1}. [{meta['source_type']}] {meta['title']} — dist: {dist:.4f}")

Search: 'budget financial planning' (PDFs only)

1. [pdf] Annual Budget Proposal 2025 — dist: 0.6224
2. [pdf] Izvještaj o upisu studenata 2024 — dist: 0.8090
3. [pdf] Ugovor o djelu - Predavač — dist: 0.8139
4. [pdf] Smjernice za pisanje završnog rada — dist: 0.8388
5. [pdf] Meeting Minutes - IT Department — dist: 0.8751


In [24]:
# Search only Bosnian-language documents
results = collection.query(
    query_texts=["university education students"],
    n_results=5,
    where={"language": "bs"},
)

print("Search: 'university education students' (Bosnian only)\n")
for i in range(len(results["ids"][0])):
    meta = results["metadatas"][0][i]
    dist = results["distances"][0][i]
    print(f"{i+1}. [{meta['source_type']}] {meta['title']} — dist: {dist:.4f}")
    print(f"   {results['documents'][0][i][:100]}...")
    print()

Search: 'university education students' (Bosnian only)

1. [web] klix.ba - IBU na listi najboljih univerziteta — dist: 0.5695
   International Burch University ponovo se našao na listi najboljih univerziteta u regionu prema Times...

2. [pdf] Izvještaj o upisu studenata 2024 — dist: 0.6988
   Na Fakultet inženjerskih i prirodnih nauka upisano je 120 studenata u akademskoj godini 2024/2025. O...

3. [pdf] Ugovor o djelu - Predavač — dist: 0.7323
   Ugovor o djelu između International Burch University i vanjskog saradnika za izvođenje nastave na pr...

4. [web] avaz.ba - Sajam zapošljavanja Sarajevo — dist: 0.7404
   U Sarajevu je održan sajam zapošljavanja na kojem je učestvovalo 40 kompanija iz IT sektora. Najtraž...

5. [pdf] Pravilnik o akademskom integritetu — dist: 0.7461
   Studenti su dužni poštovati pravila akademskog integriteta. Plagijat uključuje kopiranje tuđeg rada ...



In [25]:
# Combine multiple filters with $and / $or
results = collection.query(
    query_texts=["meeting project planning"],
    n_results=5,
    where={
        "$and": [
            {"source_type": {"$in": ["audio", "image"]}},  # audio or image sources
            {"confidence": {"$gte": 0.80}},                 # confidence >= 80%
        ]
    },
)

print("Search: 'meeting project planning' (audio/image only, confidence >= 80%)\n")
for i in range(len(results["ids"][0])):
    meta = results["metadatas"][0][i]
    dist = results["distances"][0][i]
    print(f"{i+1}. [{meta['source_type']}] {meta['title']}")
    print(f"   Confidence: {meta['confidence']}, Distance: {dist:.4f}")

Search: 'meeting project planning' (audio/image only, confidence >= 80%)

1. [audio] voice_memo_meeting_notes
   Confidence: 0.84, Distance: 0.7029
2. [image] parking_ticket_scan
   Confidence: 0.88, Distance: 0.8082
3. [image] receipt_scan_001
   Confidence: 0.82, Distance: 0.8317
4. [image] product_label_scan
   Confidence: 0.85, Distance: 0.8699
5. [audio] customer_call_support
   Confidence: 0.87, Distance: 0.9448


### ChromaDB Filter Operators

| Operator | Example | Meaning |
|----------|---------|---------|
| `$eq` | `{"language": {"$eq": "en"}}` | Equals (default) |
| `$ne` | `{"language": {"$ne": "bs"}}` | Not equals |
| `$gt`, `$gte` | `{"confidence": {"$gte": 0.9}}` | Greater than (or equal) |
| `$lt`, `$lte` | `{"word_count": {"$lt": 50}}` | Less than (or equal) |
| `$in` | `{"source_type": {"$in": ["pdf","web"]}}` | In list |
| `$nin` | `{"source_type": {"$nin": ["image"]}}` | Not in list |
| `$and` | `{"$and": [filter1, filter2]}` | Both conditions |
| `$or` | `{"$or": [filter1, filter2]}` | Either condition |

## 9. Head-to-Head: Keyword vs Semantic Search

Let's run several queries and compare the results:

In [26]:
test_queries = [
    "how much money was spent on office supplies",
    "food restaurant prices in Sarajevo",  
    "Python programming libraries for data analysis",
    "student enrollment numbers at the university",
    "problems with internet connection speed",
    "artificial intelligence and natural language processing",
]

print("=" * 80)
print("KEYWORD vs SEMANTIC SEARCH COMPARISON")
print("=" * 80)

for query in test_queries:
    kw_results = keyword_search(df, query, top_k=3)
    
    chroma_results = collection.query(
        query_texts=[query],
        n_results=3,
    )
    
    print(f"\nQuery: '{query}'")
    print("-" * 60)
    
    print("  KEYWORD:")
    if kw_results:
        for i, r in enumerate(kw_results, 1):
            print(f"    {i}. {r['title']} ({r['score']:.0%})")
    else:
        print("    (no results)")
    
    print("  SEMANTIC:")
    for i in range(min(3, len(chroma_results["ids"][0]))):
        meta = chroma_results["metadatas"][0][i]
        dist = chroma_results["distances"][0][i]
        print(f"    {i}. {meta['title']} (dist: {dist:.3f})")
    
    print()

KEYWORD vs SEMANTIC SEARCH COMPARISON

Query: 'how much money was spent on office supplies'
------------------------------------------------------------
  KEYWORD:
    1. Research Paper Draft - NLP (25%)
    2. interview_student_project (25%)
    3. podcast_episode_12_ai (25%)
  SEMANTIC:
    0. Receipt - Office Supplies (dist: 0.744)
    1. Annual Budget Proposal 2025 (dist: 0.753)
    2. Quarterly Sales Summary Q3 (dist: 0.770)


Query: 'food restaurant prices in Sarajevo'
------------------------------------------------------------
  KEYWORD:
    1. Sarajevo Tourism Report 2024 (40%)
    2. Invoice INV-00123 (40%)
    3. Receipt - Office Supplies (40%)
  SEMANTIC:
    0. Sarajevo Tourism Report 2024 (dist: 0.488)
    1. parking_ticket_scan (dist: 0.576)
    2. Receipt - Office Supplies (dist: 0.580)


Query: 'Python programming libraries for data analysis'
------------------------------------------------------------
  KEYWORD:
    1. lecture_recording_week4 (67%)
    2. Python Works

### Observations

Look at the results above and notice the patterns:

1. **Keyword wins** when the query uses the exact words that appear in documents (e.g., specific product names, invoice numbers)
2. **Semantic wins** when the query describes a concept with different words than the document uses
3. **Both struggle** with very short documents or documents in a different language from the query

This is why **hybrid search** exists.

## 10. Hybrid Search — Best of Both Worlds

Hybrid search combines keyword scores and semantic scores. The simplest approach:

1. Run keyword search → get a ranked list
2. Run semantic search → get a ranked list
3. **Combine the rankings** using Reciprocal Rank Fusion (RRF)

### Reciprocal Rank Fusion (RRF)

For each document, its RRF score is:

$$\text{RRF}(d) = \sum_{r \in \text{ranklists}} \frac{1}{k + \text{rank}_r(d)}$$

Where `k` is a constant (typically 60) that prevents top-ranked documents from dominating too much.

In [27]:
def hybrid_search(query, documents_df, collection, top_k=5, k=60):
    """
    Hybrid search combining keyword and semantic results using RRF.
    
    Args:
        query: search query string
        documents_df: pandas DataFrame with documents
        collection: ChromaDB collection
        top_k: number of results to return
        k: RRF constant (default 60)
    
    Returns:
        list of results with combined scores
    """
    # 1. Keyword search — get ranked list
    kw_results = keyword_search(documents_df, query, top_k=20)
    kw_ranking = {}
    for rank, r in enumerate(kw_results):
        kw_ranking[r["title"]] = rank + 1  # 1-based ranking
    
    # 2. Semantic search via ChromaDB — get ranked list  
    sem_results = collection.query(
        query_texts=[query],
        n_results=20,
    )
    sem_ranking = {}
    for rank in range(len(sem_results["ids"][0])):
        title = sem_results["metadatas"][0][rank]["title"]
        sem_ranking[title] = rank + 1  # 1-based ranking
    
    # 3. Combine with RRF
    all_titles = set(list(kw_ranking.keys()) + list(sem_ranking.keys()))
    
    rrf_scores = {}
    for title in all_titles:
        score = 0.0
        if title in kw_ranking:
            score += 1.0 / (k + kw_ranking[title])
        if title in sem_ranking:
            score += 1.0 / (k + sem_ranking[title])
        rrf_scores[title] = score
    
    # Sort by RRF score (highest first)
    ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    
    results = []
    for title, rrf_score in ranked[:top_k]:
        # Find the document in our DataFrame
        doc_row = documents_df[documents_df["title"] == title].iloc[0]
        results.append({
            "title": title,
            "source_type": doc_row["source_type"],
            "rrf_score": rrf_score,
            "keyword_rank": kw_ranking.get(title, "—"),
            "semantic_rank": sem_ranking.get(title, "—"),
            "text_preview": doc_row["extracted_text"][:100] + "..."
        })
    
    return results

In [28]:
# Test hybrid search
query = "artificial intelligence and natural language processing"

print(f"Hybrid search: '{query}'\n")
results = hybrid_search(query, df, collection)

print(f"{'Rank':<5} {'Title':<45} {'RRF':>6} {'KW':>5} {'Sem':>5}")
print("-" * 70)
for i, r in enumerate(results, 1):
    kw = str(r['keyword_rank'])
    sem = str(r['semantic_rank'])
    print(f"{i:<5} {r['title'][:44]:<45} {r['rrf_score']:.4f} {kw:>5} {sem:>5}")
    print(f"      {r['text_preview']}")
    print()

Hybrid search: 'artificial intelligence and natural language processing'

Rank  Title                                            RRF    KW   Sem
----------------------------------------------------------------------
1     Research Paper Draft - NLP                    0.0325     2     1
      This paper presents a novel approach to sentiment analysis for Bosnian language text using transform...

2     podcast_episode_12_ai                         0.0318     4     2
      The rise of large language models has completely changed how we think about unstructured data. Previ...

3     Python Workshop Handout                       0.0311     1     8
      Introduction to Python for data processing. Python is the most popular language for data science and...

4     Medium - Building Semantic Search with Pytho  0.0308     6     4
      Building a semantic search engine is simpler than you think. Step 1: Choose your embedding model. Fo...

5     lecture_recording_week4                       0.03

In [29]:
# Compare all three approaches side by side for multiple queries
comparison_queries = [
    "how to handle large files in Python",
    "salary and employment in IT sector",
    "student grades and academic performance",
]

for query in comparison_queries:
    print(f"\n{'='*70}")
    print(f"Query: '{query}'")
    print(f"{'='*70}")
    
    # Keyword
    kw = keyword_search(df, query, top_k=3)
    print("\n  KEYWORD TOP-3:")
    for i, r in enumerate(kw[:3], 1):
        print(f"    {i}. {r['title']}")
    if not kw:
        print("    (no results)")
    
    # Semantic
    sem = collection.query(query_texts=[query], n_results=3)
    print("\n  SEMANTIC TOP-3:")
    for i in range(min(3, len(sem["ids"][0]))):
        print(f"    {i+1}. {sem['metadatas'][0][i]['title']}")
    
    # Hybrid
    hyb = hybrid_search(query, df, collection, top_k=3)
    print("\n  HYBRID TOP-3:")
    for i, r in enumerate(hyb, 1):
        print(f"    {i}. {r['title']} (kw:{r['keyword_rank']}, sem:{r['semantic_rank']})")


Query: 'how to handle large files in Python'

  KEYWORD TOP-3:
    1. lecture_recording_week4
    2. podcast_episode_12_ai
    3. lecture_recording_embeddings

  SEMANTIC TOP-3:
    1. Python Workshop Handout
    2. lecture_recording_week4
    3. blog - ChromaDB Production Tips

  HYBRID TOP-3:
    1. lecture_recording_week4 (kw:1, sem:2)
    2. podcast_episode_12_ai (kw:2, sem:6)
    3. Python Workshop Handout (kw:9, sem:1)

Query: 'salary and employment in IT sector'

  KEYWORD TOP-3:
    1. Sarajevo Tourism Report 2024
    2. Quarterly Sales Summary Q3
    3. Annual Budget Proposal 2025

  SEMANTIC TOP-3:
    1. Annual Budget Proposal 2025
    2. klix.ba - Razvoj IT sektora u BiH
    3. Meeting Minutes - IT Department

  HYBRID TOP-3:
    1. Annual Budget Proposal 2025 (kw:3, sem:1)
    2. Quarterly Sales Summary Q3 (kw:2, sem:5)
    3. Contract Agreement v3 (kw:5, sem:7)

Query: 'student grades and academic performance'

  KEYWORD TOP-3:
    1. Security Audit Report October 2024
 

## 11. Filtering by Document Text

ChromaDB also supports **document-level** filtering — searching within the text content itself:

In [30]:
# Find documents that CONTAIN a specific word AND are semantically similar
results = collection.query(
    query_texts=["technology investment"],
    n_results=5,
    where_document={"$contains": "KM"},  # Only docs mentioning KM (currency)
)

print("Semantic search for 'technology investment' where text contains 'KM':\n")
for i in range(len(results["ids"][0])):
    meta = results["metadatas"][0][i]
    dist = results["distances"][0][i]
    print(f"{i+1}. [{meta['source_type']}] {meta['title']} — dist: {dist:.4f}")

Semantic search for 'technology investment' where text contains 'KM':

1. [pdf] Annual Budget Proposal 2025 — dist: 0.4969
2. [pdf] Quarterly Sales Summary Q3 — dist: 0.6723
3. [web] oslobodjenje.ba - Digitalizacija obrazovanja — dist: 0.7611
4. [web] klix.ba - Razvoj IT sektora u BiH — dist: 0.7862
5. [pdf] Meeting Minutes - IT Department — dist: 0.8198


## 12. Putting It All Together: A Search Function

Let's build a complete, reusable search function for our pipeline:

In [31]:
def search_pipeline(query, collection, mode="hybrid", 
                    source_filter=None, language_filter=None, top_k=5):
    """
    Complete search function for the VisitBosnia pipeline.
    
    Args:
        query: natural language search query
        collection: ChromaDB collection
        mode: 'keyword', 'semantic', or 'hybrid'
        source_filter: filter by source type (e.g., 'pdf', 'audio')
        language_filter: filter by language ('en' or 'bs')
        top_k: number of results
    
    Returns:
        formatted search results
    """
    # Build metadata filter
    where_filter = None
    conditions = []
    if source_filter:
        conditions.append({"source_type": source_filter})
    if language_filter:
        conditions.append({"language": language_filter})
    
    if len(conditions) == 1:
        where_filter = conditions[0]
    elif len(conditions) > 1:
        where_filter = {"$and": conditions}
    
    if mode == "semantic" or mode == "hybrid":
        results = collection.query(
            query_texts=[query],
            n_results=top_k,
            where=where_filter,
        )
        
        formatted = []
        for i in range(len(results["ids"][0])):
            formatted.append({
                "rank": i + 1,
                "title": results["metadatas"][0][i]["title"],
                "source_type": results["metadatas"][0][i]["source_type"],
                "language": results["metadatas"][0][i]["language"],
                "distance": results["distances"][0][i],
                "text_preview": results["documents"][0][i][:150] + "...",
            })
        return formatted
    
    elif mode == "keyword":
        # For keyword mode, use our manual function
        kw_results = keyword_search(df, query, top_k=top_k)
        return [{
            "rank": i + 1,
            "title": r["title"],
            "source_type": r["source_type"],
            "score": r["score"],
            "text_preview": r["text_preview"],
        } for i, r in enumerate(kw_results)]

In [32]:
# Interactive-style search demo
queries_to_test = [
    {"query": "What security issues were found?", "mode": "semantic"},
    {"query": "invoices and payments", "mode": "semantic", "source_filter": "pdf"},
    {"query": "student projects", "mode": "semantic", "language_filter": "en"},
    {"query": "food and restaurants", "mode": "semantic"},
]

for q in queries_to_test:
    query = q["query"]
    mode = q.get("mode", "semantic")
    sf = q.get("source_filter")
    lf = q.get("language_filter")
    
    filters_str = ""
    if sf: filters_str += f", source={sf}"
    if lf: filters_str += f", lang={lf}"
    
    print(f"🔍 '{query}' [{mode}{filters_str}]")
    results = search_pipeline(query, collection, mode=mode, 
                               source_filter=sf, language_filter=lf)
    for r in results[:3]:
        print(f"   {r['rank']}. [{r['source_type']}] {r['title']} — dist: {r['distance']:.3f}")
    print()

🔍 'What security issues were found?' [semantic]
   1. [pdf] Security Audit Report October 2024 — dist: 0.483
   2. [pdf] Meeting Minutes - IT Department — dist: 0.636
   3. [audio] team_standup_recording — dist: 0.797

🔍 'invoices and payments' [semantic, source=pdf]
   1. [pdf] Invoice INV-00123 — dist: 0.556
   2. [pdf] Receipt - Office Supplies — dist: 0.802
   3. [pdf] Quarterly Sales Summary Q3 — dist: 0.812

🔍 'student projects' [semantic, lang=en]
   1. [pdf] Student Feedback Survey Results — dist: 0.619
   2. [pdf] Security Audit Report October 2024 — dist: 0.671
   3. [audio] voice_memo_meeting_notes — dist: 0.719

🔍 'food and restaurants' [semantic]
   1. [audio] student_presentation_project — dist: 0.692
   2. [image] architecture_diagram_photo — dist: 0.788
   3. [pdf] Sarajevo Tourism Report 2024 — dist: 0.812



## 13. Updating and Deleting Documents

In a real pipeline, documents get updated. ChromaDB supports CRUD operations:

In [33]:
# Check current count
print(f"Documents in collection: {collection.count()}")

# Add a new document
collection.add(
    ids=["new_doc_001"],
    documents=["The new AI laboratory at IBU was inaugurated in November 2024, "
               "featuring GPU workstations for deep learning research and student projects."],
    metadatas=[{
        "source_type": "web",
        "title": "IBU AI Lab Opening",
        "language": "en",
        "word_count": 20,
        "confidence": 0.95,
        "processing_date": "2024-11-15",
    }],
)
print(f"After adding: {collection.count()}")

# Search for it
results = collection.query(query_texts=["machine learning GPU computing"], n_results=3)
print(f"\nSearch 'machine learning GPU computing':")
for i in range(len(results["ids"][0])):
    print(f"  {results['metadatas'][0][i]['title']} — dist: {results['distances'][0][i]:.3f}")

Documents in collection: 43
After adding: 44

Search 'machine learning GPU computing':
  IBU AI Lab Opening — dist: 0.541
  Python Workshop Handout — dist: 0.660
  voice_memo_meeting_notes — dist: 0.682


In [34]:
# Update a document's metadata
collection.update(
    ids=["new_doc_001"],
    metadatas=[{
        "source_type": "web",
        "title": "IBU AI Lab Opening",
        "language": "en",
        "word_count": 20,
        "confidence": 0.98,  # Updated confidence
        "processing_date": "2024-11-15",
    }],
)

# Delete a document
collection.delete(ids=["new_doc_001"])
print(f"After deleting: {collection.count()}")

After deleting: 43


## Summary

### What We Learned

| Concept | Tool / Method | Key Function |
|---------|--------------|--------------|
| **Keyword search** | Python string matching | Count word overlaps |
| **Embeddings** | `sentence-transformers` | `model.encode(text)` |
| **Cosine similarity** | NumPy | `np.dot(a,b) / (norm(a)*norm(b))` |
| **Vector database** | ChromaDB | `collection.query()` |
| **Metadata filtering** | ChromaDB `where` | `{"source_type": "pdf"}` |
| **Hybrid search** | RRF fusion | Combine keyword + semantic ranks |

### Pipeline Update

```
Acquire → Extract → Store → Explore → Clean → ★ Search ★ → Visualize → Deploy
                                                  ↑
                                        You are here!
```

### Next Week

**Week 12: Visualization** — We'll create interactive charts and dashboards to visualize our search results and document collection using Plotly and Dash.

---

### 📚 Reading

| Source | Link |
|--------|------|
| Vicki Boykis — "What are Embeddings?" | https://vickiboykis.com/what_are_embeddings/ |
| sentence-transformers — Quickstart | https://www.sbert.net/docs/quickstart.html |
| sentence-transformers — Semantic Search | https://www.sbert.net/examples/applications/semantic-search/README.html |
| ChromaDB — Getting Started | https://docs.trychroma.com/docs/overview/getting-started |
| ChromaDB — Querying Collections | https://docs.trychroma.com/docs/querying-collections/querying |